In [2]:
from openai import OpenAI
client = OpenAI()

resp = client.chat.completions.create(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Say hello"}],
)
print(resp.choices[0].message.content)

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************FsEA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# Paths to deduped parquet files
base_dir = Path("../data/Postings")

files = [
    base_dir / "SG-2022-WITHOUT-REPOSTS-W60D.parquet",
    base_dir / "SG-2023-WITHOUT-REPOSTS-W60D.parquet",
    base_dir / "SG-2024-WITHOUT-REPOSTS-W60D.parquet",
]

# Read and concatenate
data_df = pd.concat(
    [pd.read_parquet(f) for f in files],
    ignore_index=True
)

print("Combined data_df shape:", data_df.shape)
print("Years covered:", sorted(data_df["post_date"].dt.year.unique()))

Combined data_df shape: (1492675, 41)
Years covered: [2022, 2023, 2024]


In [3]:
recruitment_industries = [
    'Recruitment and Staffing Services',
    'Employment and Staffing Services',
    'Employment and Recruitment Services',
    'Human Resources and Recruitment Services',
    'Online Employment Platforms',
    'Human Resources and Workforce Solutions',
    'Business Process Outsourcing Services'
]

In [4]:
d_unique_filtered = data_df[~data_df['rics_k400'].isin(recruitment_industries)].copy()

## Using POSITION-LEVEL DATA, splitting treatment and control based on whether firms have EVER hired GENAI INTEGRATORS

1. Identify firms that have ever hired GenAI integrators from the POSTINGS dataset;
2. Identify positions from these firms that have ever hired GenAI integrators, and label these positions as treated;
3. Run DiD on POSITION-LEVEL data to compare the evolution of number of posts for treated vs. non-treated firms

In [8]:
import os
import re
import glob
import pandas as pd
import numpy as np

def build_positions_panel_treated_by_ever_genai_posting(
    d_unique: pd.DataFrame,
    positions_dir: str = r"C:\LocalOneDrive\Documents\Desktop\NUS\Y4S1\FYP\causal-impact-GenAI\data\Positions",
    company_positions_filename: str = "company_positions.csv",
    recruitment_industries: list | None = None,
    # ---- postings (treatment identification)
    posting_firm_col: str = "company",
    posting_desc_col: str = "description",
    posting_date_col: str = "post_date",
    posting_ind_col: str = "rics_k400",
    postings_min_qtr: str = "2022Q1",
    postings_max_qtr: str | None = None,
    # ---- positions (schema fixed as requested)
    pos_date_col: str = "startdate",
    pos_id_col: str = "position_id",
    cp_join_col: str = "position_id",
    cp_company_col: str = "company_name",
    # ---- panel settings
    panel_min_qtr: str = "2022Q1",
    panel_max_qtr: str = "2025Q2",
    treat_qtr: str = "2022Q4",
    out_path: str = "firm_quarter_positions_panel_treated_by_ever_genai_posting_2022Q1_2025Q2.dta",
):
    """
    1) Identify treated firms from postings: treated=1 if firm EVER has a posting mentioning GenAI keywords (within postings_min_qtr+ and after recruitment exclusion if provided).
    2) Load positions files (2022Q1..2024Q4 + 2025_positions*), merge with company_positions.csv on position_id -> company_name.
    3) Build balanced firm-quarter panel of unique positions counts.
    4) Assign treated (ever GenAI posting firms) + post (>=2022Q4 inclusive), save to .dta.

    Prints:
      - how many treated firms + % treated (within the postings sample used for treatment construction)
      - how many treated firms + % treated among firms appearing in positions panel
      - % of postings treated (share of posting rows in the postings sample belonging to treated firms)
    """

    # ==================================================
    # A) Identify treated firms from postings d_unique
    # ==================================================
    print("A) Building treated-firm indicator from postings...")

    d = d_unique.copy()
    before_all = len(d)

    # optional: exclude recruitment industries
    if recruitment_industries is not None and posting_ind_col in d.columns:
        before = len(d)
        d = d[~d[posting_ind_col].isin(recruitment_industries)].copy()
        print(f"  - Excluded recruitment industries: {before - len(d):,} rows removed")
    elif recruitment_industries is not None:
        print(f"  - Warning: '{posting_ind_col}' not in postings; skipping recruitment exclusion")

    # dates -> quarters -> window
    d[posting_date_col] = pd.to_datetime(d[posting_date_col], errors="coerce")
    d = d.dropna(subset=[posting_date_col]).copy()
    d["qtr"] = d[posting_date_col].dt.to_period("Q")

    d = d[d["qtr"] >= pd.Period(postings_min_qtr, freq="Q")].copy()
    if postings_max_qtr:
        d = d[d["qtr"] <= pd.Period(postings_max_qtr, freq="Q")].copy()

    print(f"  - Postings rows after date + filters: {len(d):,} (from {before_all:,})")
    print(f"  - Postings quarter range: {d['qtr'].min()} to {d['qtr'].max()}")

    genai_keywords = [
        "copilot", "claude", "gemini", "large language model", "llm",
        "generative ai", "chatgpt", "gen ai", "gpt", "langchain", "rag",
        "retrieval-augmented generation", "vector embedding", "vector database",
        "transformer", "prompt engineering", "prompt design", "llamaindex",
        "pinecone", "weaviate", "milvus", "openai api", "anthropic",
        "azure openai", "vertex ai", "huggingface", "retrievalqa"
    ]
    pattern = re.compile("|".join(genai_keywords), re.IGNORECASE)

    d["genai_mention"] = d[posting_desc_col].fillna("").str.contains(pattern).astype(int)

    firm_treat = (
        d.groupby(posting_firm_col)["genai_mention"]
        .max()
        .reset_index()
        .rename(columns={"genai_mention": "treated"})
    )

    treated_firms = set(firm_treat.loc[firm_treat["treated"] == 1, posting_firm_col].astype(str))

    n_firms_postings = firm_treat.shape[0]
    n_treated_postings = int(firm_treat["treated"].sum())
    pct_treated_postings_firms = (n_treated_postings / n_firms_postings) if n_firms_postings > 0 else np.nan

    treated_postings_rows = d[posting_firm_col].astype(str).isin(treated_firms).mean() if len(d) > 0 else np.nan

    print(f"  - Treated firms (postings sample): {n_treated_postings:,} / {n_firms_postings:,} ({pct_treated_postings_firms:.2%})")
    print(f"  - Treated postings rows (postings sample): {treated_postings_rows:.2%}")

    # ==================================================
    # B) Load positions files + merge company_positions
    # ==================================================
    print("\nB) Loading positions files and merging company_positions.csv...")

    company_positions_path = os.path.join(positions_dir, company_positions_filename)
    if not os.path.exists(company_positions_path):
        raise FileNotFoundError(f"Missing company_positions.csv at: {company_positions_path}")

    cp = pd.read_csv(company_positions_path)

    required_cp_cols = {cp_join_col, cp_company_col}
    missing_cp = required_cp_cols - set(cp.columns)
    if missing_cp:
        raise ValueError(f"company_positions.csv missing required columns: {missing_cp}. Found columns: {list(cp.columns)[:30]}")

    files_2022_2024 = sorted(glob.glob(os.path.join(positions_dir, "2022Q*_positions.csv"))) + \
                     sorted(glob.glob(os.path.join(positions_dir, "2023Q*_positions.csv"))) + \
                     sorted(glob.glob(os.path.join(positions_dir, "2024Q*_positions.csv")))
    files_2025 = sorted(glob.glob(os.path.join(positions_dir, "2025_positions*.csv")))

    pos_files = files_2022_2024 + files_2025
    if len(pos_files) == 0:
        raise FileNotFoundError(f"No position files found in {positions_dir} with expected patterns.")

    print(f"  - Found {len(pos_files)} positions files")
    print(f"  - First 3 files: {[os.path.basename(x) for x in pos_files[:3]]}")

    pos_list = []
    for fp in pos_files:
        tmp = pd.read_csv(fp)
        tmp["_source_file"] = os.path.basename(fp)
        pos_list.append(tmp)

    pos = pd.concat(pos_list, ignore_index=True)
    print(f"  - Loaded positions rows (raw concat): {len(pos):,}")

    required_pos_cols = {pos_id_col, pos_date_col}
    missing_pos = required_pos_cols - set(pos.columns)
    if missing_pos:
        raise ValueError(f"Positions files missing required columns: {missing_pos}. Found columns: {list(pos.columns)[:30]}")

    # merge company onto positions
    pos[pos_id_col] = pos[pos_id_col].astype(str)
    cp[cp_join_col] = cp[cp_join_col].astype(str)

    cp_key_dups = cp[cp_join_col].duplicated().sum()
    if cp_key_dups > 0:
        print(f"  - Warning: company_positions has {cp_key_dups:,} duplicate join keys; dropping duplicates for merge")
    cp_small = cp[[cp_join_col, cp_company_col]].drop_duplicates()

    pos = pos.merge(
        cp_small,
        left_on=pos_id_col,
        right_on=cp_join_col,
        how="left",
        validate="m:1"
    )

    pos = pos.rename(columns={cp_company_col: "company"})
    pos["company"] = pos["company"].astype(str)

    unmatched = pos["company"].isna().mean() if "company" in pos.columns else np.nan
    print(f"  - Merge complete. Unmatched company share: {unmatched:.2%}")

    # dates -> qtr -> filter to panel window
    pos[pos_date_col] = pd.to_datetime(pos[pos_date_col], errors="coerce")
    before_dates = len(pos)
    pos = pos.dropna(subset=[pos_date_col]).copy()
    print(f"  - Dropped rows with invalid startdate: {before_dates - len(pos):,}")

    pos["qtr"] = pos[pos_date_col].dt.to_period("Q")
    panel_min = pd.Period(panel_min_qtr, freq="Q")
    panel_max = pd.Period(panel_max_qtr, freq="Q")

    before_window = len(pos)
    pos = pos[(pos["qtr"] >= panel_min) & (pos["qtr"] <= panel_max)].copy()
    print(f"  - Positions rows after quarter window [{panel_min_qtr}, {panel_max_qtr}]: {len(pos):,} (removed {before_window - len(pos):,})")
    print(f"  - Positions quarter range (after filter): {pos['qtr'].min()} to {pos['qtr'].max()}")

    # ==================================================
    # C) Aggregate to firm-quarter positions count
    # ==================================================
    print("\nC) Aggregating to firm-quarter and balancing panel...")

    panel = (
        pos.groupby(["company", "qtr"])
        .agg(num_positions=(pos_id_col, "nunique"))
        .reset_index()
    )

    firms = panel[["company"]].drop_duplicates()
    all_qtrs = pd.period_range(panel_min, panel_max, freq="Q")
    full = firms.assign(_tmp=1).merge(pd.DataFrame({"qtr": all_qtrs, "_tmp": 1}), on="_tmp").drop(columns="_tmp")

    panel = full.merge(panel, on=["company", "qtr"], how="left")
    panel["num_positions"] = panel["num_positions"].fillna(0).astype(int)

    print(f"  - Firms in positions panel: {panel['company'].nunique():,}")
    print(f"  - Firm-quarter rows (balanced): {len(panel):,}")

    # ==================================================
    # D) Treatment + post indicators
    # ==================================================
    print("\nD) Assigning treated and post indicators...")

    panel["treated"] = panel["company"].astype(str).isin(treated_firms).astype(int)

    n_firms_positions = panel[["company", "treated"]].drop_duplicates().shape[0]
    n_treated_positions = int(panel[["company", "treated"]].drop_duplicates()["treated"].sum())
    pct_treated_positions_firms = (n_treated_positions / n_firms_positions) if n_firms_positions > 0 else np.nan

    print(f"  - Treated firms (positions panel): {n_treated_positions:,} / {n_firms_positions:,} ({pct_treated_positions_firms:.2%})")

    TREAT_QTR = pd.Period(treat_qtr, freq="Q")
    panel["post"] = (panel["qtr"] >= TREAT_QTR).astype(int)
    panel["did"] = panel["treated"] * panel["post"]

    panel["log_positions"] = np.log(panel["num_positions"] + 1)

    # Stata-aligned time vars
    panel["year"] = panel["qtr"].dt.year
    panel["qtr_num"] = panel["qtr"].dt.quarter
    panel["tq"] = (panel["year"] - 1960) * 4 + (panel["qtr_num"] - 1)

    panel["firm_id"] = panel["company"].astype("category").cat.codes + 1
    panel["qtr_str"] = panel["qtr"].astype(str)

    panel = panel.drop(columns="qtr").sort_values(["firm_id", "year", "qtr_num"])

    # ==================================================
    # E) Save
    # ==================================================
    panel.to_stata(out_path, write_index=False, version=118)

    print("\nE) Saved outputs")
    print(f"  - Saved positions panel: {out_path}")
    print(f"  - Quarters in panel: {panel_min_qtr} to {panel_max_qtr} (inclusive)")
    print(f"  - Post starts at: {treat_qtr} (inclusive)")
    print(f"  - Outcome: num_positions, log_positions")
    print(f"  - Treatment: treated (ever GenAI posting firms)")

    return panel, firm_treat

In [9]:
panel_pos, firm_treat = build_positions_panel_treated_by_ever_genai_posting(
    d_unique=d_unique_filtered,
    recruitment_industries=recruitment_industries,  # or None if you don't want this filter
    out_path=r"C:\LocalOneDrive\Documents\Desktop\NUS\Y4S1\FYP\causal-impact-GenAI\data\Positions\firm_quarter_positions_panel_treated_2022Q1_2025Q2.dta"
)

A) Building treated-firm indicator from postings...
  - Excluded recruitment industries: 0 rows removed
  - Postings rows after date + filters: 997,690 (from 997,690)
  - Postings quarter range: 2022Q1 to 2024Q4
  - Treated firms (postings sample): 15,568 / 37,442 (41.58%)
  - Treated postings rows (postings sample): 85.28%

B) Loading positions files and merging company_positions.csv...
  - Found 13 positions files
  - First 3 files: ['2022Q1_positions.csv', '2022Q2_positions.csv', '2022Q3_positions.csv']
  - Loaded positions rows (raw concat): 3,095,883
  - Merge complete. Unmatched company share: 0.00%
  - Dropped rows with invalid startdate: 0
  - Positions rows after quarter window [2022Q1, 2025Q2]: 1,458,920 (removed 1,636,963)
  - Positions quarter range (after filter): 2022Q1 to 2025Q2

C) Aggregating to firm-quarter and balancing panel...
  - Firms in positions panel: 96,154
  - Firm-quarter rows (balanced): 1,346,156

D) Assigning treated and post indicators...
  - Treated fi

## Splitting treatment and control based on terciles instead of using a median split strategy to test for robustness

In [7]:
def build_occ_ind_quarter_panel_baseline_no_recruitment(
    d_unique: pd.DataFrame,
    occ_level_path: str,
    recruitment_industries: list,
    exposure_col: str = "human_rating_beta",
    date_col: str = "post_date",
    onet_postings_col: str = "onet_code",
    onet_occ_col: str = "O*NET-SOC Code",
    ind_col: str = "rics_k50",
    recruit_ind_col: str = "rics_k400",
    firm_col: str = "ultimate_parent_company_name",
    company_col: str = "company",  # kept for signature compatibility (unused here)
    exclude_firms: list[str] | None = None,
    treat_date: str = "2022-11-30",
    out_path: str = "occ_ind_quarter_panel_highlow_human_beta_no_recruit_baseline.dta",
    min_qtr: str = "2022Q1",
    max_qtr: str | None = None,
):
    d = d_unique.copy()

    # --------------------------------------------------
    # 0) Exclude recruitment-related industry postings
    # --------------------------------------------------
    if recruit_ind_col in d.columns:
        before = len(d)
        d_only_recruit = d[d[recruit_ind_col].isin(recruitment_industries)].copy()
        d = d[~d[recruit_ind_col].isin(recruitment_industries)].copy()
        removed_share = len(d_only_recruit) / before if before else 0
        print(f"Excluded recruitment industries: {len(d_only_recruit)} rows ({removed_share:.2%})")
    else:
        print(f"Warning: '{recruit_ind_col}' not found; no recruitment-industry filtering applied.")

    # --------------------------------------------------
    # 0b) Optional firm exclusion (unchanged behavior)
    # --------------------------------------------------
    if exclude_firms:
        if firm_col not in d.columns:
            print(f"Warning: firm_col '{firm_col}' not found; firm exclusion skipped.")
        else:
            before = len(d)
            d_firm = d[firm_col].astype(str).str.strip().str.casefold()
            exclude_set = {str(x).strip().casefold() for x in exclude_firms}
            mask_excl = d_firm.isin(exclude_set)
            d = d.loc[~mask_excl].copy()
            removed = int(mask_excl.sum())
            print(f"Excluded firms: {removed} rows ({removed / before:.2%})")

    # --------------------------------------------------
    # 1) Dates
    # --------------------------------------------------
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d = d.dropna(subset=[date_col]).copy()
    treat_dt = pd.Timestamp(treat_date)

    # --------------------------------------------------
    # 2) Load exposure table
    # --------------------------------------------------
    occ = pd.read_csv(occ_level_path)
    if exposure_col not in occ.columns:
        raise ValueError(f"{exposure_col} not found in occ_level.csv")

    d[onet_postings_col] = d[onet_postings_col].astype(str).str.strip()
    occ[onet_occ_col] = occ[onet_occ_col].astype(str).str.strip()

    occ_sub = occ[[onet_occ_col, exposure_col]].rename(columns={onet_occ_col: onet_postings_col})

    d = d.merge(occ_sub, on=onet_postings_col, how="left", indicator=True)
    d = d[d["_merge"] == "both"].drop(columns=["_merge"]).copy()

    # --------------------------------------------------
    # 3) Pre-treatment terciles (occupation-level, pre-treat_date)
    #    Keep only bottom tercile (control) and top tercile (treated)
    # --------------------------------------------------
    d_pre = d[d[date_col] < treat_dt]
    occ_pre = d_pre[[onet_postings_col, exposure_col]].drop_duplicates()

    # Tercile cutoffs based on pre-period occupation exposure distribution
    q33 = float(occ_pre[exposure_col].quantile(1/3))
    q67 = float(occ_pre[exposure_col].quantile(2/3))

    occ_treat = (
        d[[onet_postings_col, exposure_col]]
        .drop_duplicates()
        .assign(
            treated=lambda x: np.select(
                [x[exposure_col] <= q33, x[exposure_col] >= q67],
                [0, 1],
                default=np.nan
            )
        )
        [[onet_postings_col, "treated"]]
    )

    # Drop middle-tercile occupations (treated is NaN)
    keep_occs = occ_treat.dropna(subset=["treated"])[onet_postings_col].unique()
    d = d[d[onet_postings_col].isin(keep_occs)].copy()

    # Keep occ_treat aligned to retained occupations only
    occ_treat = occ_treat.dropna(subset=["treated"]).copy()
    occ_treat["treated"] = occ_treat["treated"].astype(int)

    # --------------------------------------------------
    # 4) Quarter variables
    # --------------------------------------------------
    d["qtr"] = d[date_col].dt.to_period("Q")
    d = d[d["qtr"] >= pd.Period(min_qtr, freq="Q")]
    if max_qtr:
        d = d[d["qtr"] <= pd.Period(max_qtr, freq="Q")]

    # --------------------------------------------------
    # 5) Aggregate to occ–ind–quarter
    # --------------------------------------------------
    panel = (
        d.groupby([onet_postings_col, ind_col, "qtr"])
        .agg(num_postings=("job_id", "count"))
        .reset_index()
    )

    # --------------------------------------------------
    # 6) Balance panel
    # --------------------------------------------------
    occ_ind_pairs = panel[[onet_postings_col, ind_col]].drop_duplicates()
    all_qtrs = pd.period_range(panel["qtr"].min(), panel["qtr"].max(), freq="Q")

    full = (
        occ_ind_pairs.assign(_tmp=1)
        .merge(pd.DataFrame({"qtr": all_qtrs, "_tmp": 1}), on="_tmp")
        .drop(columns="_tmp")
    )

    panel = full.merge(panel, on=[onet_postings_col, ind_col, "qtr"], how="left")
    panel["num_postings"] = panel["num_postings"].fillna(0).astype(int)

    # --------------------------------------------------
    # 7) Treatment timing (2022Q4 inclusive)
    # --------------------------------------------------
    panel = panel.merge(occ_treat, on=onet_postings_col, how="left")
    panel["treated"] = panel["treated"].fillna(0).astype(int)

    TREAT_QTR = pd.Period("2022Q4", freq="Q")
    panel["post"] = (panel["qtr"] >= TREAT_QTR).astype(int)
    panel["did"] = panel["treated"] * panel["post"]
    panel["log_postings"] = np.log(panel["num_postings"] + 1)

    # --------------------------------------------------
    # 8) Stata-aligned time vars
    # --------------------------------------------------
    panel["year"] = panel["qtr"].dt.year
    panel["qtr_num"] = panel["qtr"].dt.quarter
    panel["tq"] = (panel["year"] - 1960) * 4 + (panel["qtr_num"] - 1)

    base_tq = (2022 - 1960) * 4
    event_tq = base_tq + 3

    panel["t_index"] = panel["tq"] - base_tq
    panel["relq"] = panel["tq"] - event_tq

    # --------------------------------------------------
    # 9) IDs for FE
    # --------------------------------------------------
    panel["occ_id"] = panel[onet_postings_col].astype("category").cat.codes + 1
    panel["ind_id"] = panel[ind_col].astype("category").cat.codes + 1
    panel["occ_ind_id"] = (
        panel[onet_postings_col].astype(str) + "|" + panel[ind_col].astype(str)
    ).astype("category").cat.codes + 1

    panel["qtr_str"] = panel["qtr"].astype(str)
    panel = panel.drop(columns="qtr").sort_values(["occ_ind_id", "year", "qtr_num"])

    panel.to_stata(out_path, write_index=False, version=118)

    print(f"Saved: {out_path}")
    print(f"Pre-treatment terciles ({exposure_col}): q33={q33:.4f}, q67={q67:.4f}")
    print(f"Kept occupations: {len(keep_occs)} (dropped middle tercile)")

    return panel

In [9]:
panel = build_occ_ind_quarter_panel_baseline_no_recruitment(
    d_unique=d_unique_filtered,
    occ_level_path=r"C:\LocalOneDrive\Documents\Desktop\NUS\Y4S1\FYP\causal-impact-GenAI\data\AI Exposure Scores\occ_level.csv",
    recruitment_industries=recruitment_industries,
    exposure_col="human_rating_beta",
    date_col="post_date",
    onet_postings_col="onet_code",
    onet_occ_col="O*NET-SOC Code",
    ind_col="rics_k50",
    recruit_ind_col="rics_k400",
    firm_col="ultimate_parent_company_name",
    company_col="company",
    exclude_firms=None,
    treat_date="2022-11-30",
    out_path="../data/occ_ind_quarter_panel_highlow_human_beta_no_recruit_baseline_terciles.dta",
    min_qtr="2022Q1",
    max_qtr="2024Q4",
)

Excluded recruitment industries: 0 rows (0.00%)
Saved: ../data/occ_ind_quarter_panel_highlow_human_beta_no_recruit_baseline_terciles.dta
Pre-treatment terciles (human_rating_beta): q33=0.3063, q67=0.4835
Kept occupations: 253 (dropped middle tercile)


## Attempting a continuous DiD method

In [ ]:
def build_occ_ind_quarter_panel_continuous_exposure_no_recruitment(
    d_unique: pd.DataFrame,
    occ_level_path: str,
    recruitment_industries: list,
    exposure_col: str = "human_rating_beta",
    date_col: str = "post_date",
    onet_postings_col: str = "onet_code",
    onet_occ_col: str = "O*NET-SOC Code",
    ind_col: str = "rics_k50",
    recruit_ind_col: str = "rics_k400",
    firm_col: str = "ultimate_parent_company_name",
    company_col: str = "company",  # kept for signature compatibility (unused here)
    exclude_firms: list[str] | None = None,
    treat_date: str = "2022-11-30",
    treat_qtr: str = "2022Q4",
    ref_qtr: str = "2022Q3",  # reference quarter for event-study normalization
    min_qtr: str = "2022Q1",
    max_qtr: str | None = None,
    center_exposure: bool = True,
    out_path: str = "occ_ind_quarter_panel_continuous_human_beta_no_recruit.dta",
):
    d = d_unique.copy()

    # 0) Exclude recruitment-related industry postings
    if recruit_ind_col in d.columns:
        before = len(d)
        d_only_recruit = d[d[recruit_ind_col].isin(recruitment_industries)].copy()
        d = d[~d[recruit_ind_col].isin(recruitment_industries)].copy()
        removed_share = len(d_only_recruit) / before if before else 0
        print(f"Excluded recruitment industries: {len(d_only_recruit)} rows ({removed_share:.2%})")
    else:
        print(f"Warning: '{recruit_ind_col}' not found; no recruitment-industry filtering applied.")

    # 0b) Optional firm exclusion
    if exclude_firms:
        if firm_col not in d.columns:
            print(f"Warning: firm_col '{firm_col}' not found; firm exclusion skipped.")
        else:
            before = len(d)
            d_firm = d[firm_col].astype(str).str.strip().str.casefold()
            exclude_set = {str(x).strip().casefold() for x in exclude_firms}
            mask_excl = d_firm.isin(exclude_set)
            d = d.loc[~mask_excl].copy()
            removed = int(mask_excl.sum())
            print(f"Excluded firms: {removed} rows ({removed / before:.2%})")

    # 1) Dates
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d = d.dropna(subset=[date_col]).copy()
    treat_dt = pd.Timestamp(treat_date)

    # 2) Load exposure table and merge
    occ = pd.read_csv(occ_level_path)
    if exposure_col not in occ.columns:
        raise ValueError(f"{exposure_col} not found in {occ_level_path}")

    d[onet_postings_col] = d[onet_postings_col].astype(str).str.strip()
    occ[onet_occ_col] = occ[onet_occ_col].astype(str).str.strip()
    occ_sub = occ[[onet_occ_col, exposure_col]].rename(columns={onet_occ_col: onet_postings_col})

    d = d.merge(occ_sub, on=onet_postings_col, how="left", indicator=True)
    d = d[d["_merge"] == "both"].drop(columns=["_merge"]).copy()

    d[exposure_col] = pd.to_numeric(d[exposure_col], errors="coerce")
    d = d.dropna(subset=[exposure_col]).copy()

    # 3) Center exposure using pre-period occupation distribution
    occ_all = d[[onet_postings_col, exposure_col]].drop_duplicates()
    occ_pre = d.loc[d[date_col] < treat_dt, [onet_postings_col, exposure_col]].drop_duplicates()
    pre_mean = float(occ_pre[exposure_col].mean()) if len(occ_pre) else float(occ_all[exposure_col].mean())

    if center_exposure:
        occ_all["exposure_c"] = occ_all[exposure_col] - pre_mean
        exposure_use = "exposure_c"
    else:
        occ_all["exposure_c"] = occ_all[exposure_col]
        exposure_use = exposure_col

    # 4) Quarter variables
    d["qtr"] = d[date_col].dt.to_period("Q")
    d = d[d["qtr"] >= pd.Period(min_qtr, freq="Q")]
    if max_qtr:
        d = d[d["qtr"] <= pd.Period(max_qtr, freq="Q")]

    # 5) Aggregate to occ–ind–quarter
    panel = (
        d.groupby([onet_postings_col, ind_col, "qtr"])
        .agg(num_postings=("job_id", "count"))
        .reset_index()
    )

    # 6) Balance panel (IMPORTANT FIX: start at min_qtr)
    occ_ind_pairs = panel[[onet_postings_col, ind_col]].drop_duplicates()
    minP = pd.Period(min_qtr, freq="Q")
    maxP = panel["qtr"].max()
    all_qtrs = pd.period_range(minP, maxP, freq="Q")

    full = (
        occ_ind_pairs.assign(_tmp=1)
        .merge(pd.DataFrame({"qtr": all_qtrs, "_tmp": 1}), on="_tmp")
        .drop(columns="_tmp")
    )

    panel = full.merge(panel, on=[onet_postings_col, ind_col, "qtr"], how="left")
    panel["num_postings"] = panel["num_postings"].fillna(0).astype(int)

    # 7) Merge exposure onto panel
    panel = panel.merge(occ_all[[onet_postings_col, exposure_use]], on=onet_postings_col, how="left")
    if panel[exposure_use].isna().any():
        raise ValueError("Some panel rows have missing exposure after merge (unexpected).")

    TREAT_QTR = pd.Period(treat_qtr, freq="Q")
    REF_QTR = pd.Period(ref_qtr, freq="Q")

    panel["post"] = (panel["qtr"] >= TREAT_QTR).astype(int)
    panel["log_postings"] = np.log(panel["num_postings"] + 1)

    # 8) Stata-aligned time vars
    panel["year"] = panel["qtr"].dt.year
    panel["qtr_num"] = panel["qtr"].dt.quarter
    panel["tq"] = (panel["year"] - 1960) * 4 + (panel["qtr_num"] - 1)

    base_tq = (minP.year - 1960) * 4 + (minP.quarter - 1)
    treat_tq = (TREAT_QTR.year - 1960) * 4 + (TREAT_QTR.quarter - 1)
    ref_tq = (REF_QTR.year - 1960) * 4 + (REF_QTR.quarter - 1)

    panel["t_index"] = panel["tq"] - base_tq

    # IMPORTANT FIX:
    panel["relq_event"] = panel["tq"] - treat_tq   # 0 at 2022Q4; 2022Q1 becomes -3
    panel["relq_ref"] = panel["tq"] - ref_tq       # 0 at 2022Q3 (optional)

    # 9) IDs for FE
    panel["occ_id"] = panel[onet_postings_col].astype("category").cat.codes + 1
    panel["ind_id"] = panel[ind_col].astype("category").cat.codes + 1
    panel["occ_ind_id"] = (
        panel[onet_postings_col].astype(str) + "|" + panel[ind_col].astype(str)
    ).astype("category").cat.codes + 1

    panel["qtr_str"] = panel["qtr"].astype(str)
    panel = panel.drop(columns="qtr").sort_values(["occ_ind_id", "year", "qtr_num"])

    panel.to_stata(out_path, write_index=False, version=118)

    print(f"Saved: {out_path}")
    print(f"Exposure column for Stata: {exposure_use} (centered={center_exposure})")
    print(f"Pre-period mean exposure used for centering: {pre_mean:.4f}")
    print(f"Event time column: relq_event (0={treat_qtr}) ; reference quarter is {ref_qtr} (relq_ref=0)")

    return panel, exposure_use, pre_mean

In [14]:
panel_cont, exposure_use, pre_mean = build_occ_ind_quarter_panel_continuous_exposure_no_recruitment(
    d_unique=d_unique_filtered,
    occ_level_path=r"C:\LocalOneDrive\Documents\Desktop\NUS\Y4S1\FYP\causal-impact-GenAI\data\AI Exposure Scores\occ_level.csv",
    recruitment_industries=recruitment_industries,
    exposure_col="human_rating_beta",
    date_col="post_date",
    onet_postings_col="onet_code",
    onet_occ_col="O*NET-SOC Code",
    ind_col="rics_k50",
    recruit_ind_col="rics_k400",
    firm_col="ultimate_parent_company_name",
    company_col="company",
    exclude_firms=None,
    treat_date="2022-11-30",
    treat_qtr="2022Q4",
    ref_qtr="2022Q3",
    min_qtr="2022Q1",
    max_qtr="2024Q4",
    center_exposure=True,
    out_path="../data/occ_ind_quarter_panel_continuous_human_beta_no_recruit.dta",
)
print(exposure_use, pre_mean)

Excluded recruitment industries: 0 rows (0.00%)
Saved: ../data/occ_ind_quarter_panel_continuous_human_beta_no_recruit.dta
Exposure column for Stata: exposure_c (centered=True)
Pre-period mean exposure used for centering: 0.3773
Event time column: relq_event (0=2022Q4) ; reference quarter is 2022Q3 (relq_ref=0)
exposure_c 0.3773027620899471


## Assigning treatment and control based on whether firms have ever-hired GenAI integrators

In [ ]:
import re

def build_firm_quarter_panel_genai_hiring(
    d_unique: pd.DataFrame,
    recruitment_industries: list,
    date_col: str = "post_date",
    firm_col: str = "company",
    desc_col: str = "description",
    ind_col: str = "rics_k400",
    treat_qtr: str = "2022Q4",
    min_qtr: str = "2022Q1",
    max_qtr: str | None = None,
    out_path: str = "firm_quarter_panel_genai_ever_hired_no_recruit.dta",
):
    d = d_unique.copy()

    # --------------------------------------------------
    # 0) Exclude recruitment-related postings
    # --------------------------------------------------
    if ind_col in d.columns:
        before = len(d)
        d = d[~d[ind_col].isin(recruitment_industries)].copy()
        print(f"Excluded recruitment industries: {before - len(d)} rows")
    else:
        print(f"Warning: {ind_col} not found; skipping recruitment exclusion")

    # --------------------------------------------------
    # 1) Dates
    # --------------------------------------------------
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d = d.dropna(subset=[date_col]).copy()

    d["qtr"] = d[date_col].dt.to_period("Q")
    d = d[d["qtr"] >= pd.Period(min_qtr, freq="Q")]
    if max_qtr:
        d = d[d["qtr"] <= pd.Period(max_qtr, freq="Q")]

    # --------------------------------------------------
    # 2) Detect GenAI-related hiring from descriptions
    # --------------------------------------------------
    genai_keywords = [
        "copilot", "claude", "gemini", "large language model", "llm",
        "generative ai", "chatgpt", "gen ai", "gpt", "langchain", "rag",
        "retrieval-augmented generation", "vector embedding", "vector database",
        "transformer", "prompt engineering", "prompt design", "llamaindex",
        "pinecone", "weaviate", "milvus", "openai api", "anthropic",
        "azure openai", "vertex ai", "huggingface", "retrievalqa"
    ]

    pattern = re.compile("|".join(genai_keywords), re.IGNORECASE)

    d["genai_mention"] = d[desc_col].fillna("").str.contains(pattern).astype(int)

    # --------------------------------------------------
    # 3) Ever-treated firm indicator
    # --------------------------------------------------
    firm_treat = (
        d.groupby(firm_col)["genai_mention"]
        .max()
        .reset_index()
        .rename(columns={"genai_mention": "treated"})
    )

    # --------------------------------------------------
    # 4) Aggregate to firm–quarter
    # --------------------------------------------------
    panel = (
        d.groupby([firm_col, "qtr"])
        .agg(num_postings=("job_id", "count"))
        .reset_index()
    )

    # --------------------------------------------------
    # 5) Balance firm–quarter panel
    # --------------------------------------------------
    firms = panel[[firm_col]].drop_duplicates()
    all_qtrs = pd.period_range(panel["qtr"].min(), panel["qtr"].max(), freq="Q")

    full = (
        firms.assign(_tmp=1)
        .merge(pd.DataFrame({"qtr": all_qtrs, "_tmp": 1}), on="_tmp")
        .drop(columns="_tmp")
    )

    panel = full.merge(panel, on=[firm_col, "qtr"], how="left")
    panel["num_postings"] = panel["num_postings"].fillna(0).astype(int)

    # --------------------------------------------------
    # 6) Merge treatment + post
    # --------------------------------------------------
    panel = panel.merge(firm_treat, on=firm_col, how="left")
    panel["treated"] = panel["treated"].fillna(0).astype(int)

    TREAT_QTR = pd.Period(treat_qtr, freq="Q")
    panel["post"] = (panel["qtr"] >= TREAT_QTR).astype(int)
    panel["did"] = panel["treated"] * panel["post"]
    panel["log_postings"] = np.log(panel["num_postings"] + 1)

    # --------------------------------------------------
    # 7) Stata-aligned time vars
    # --------------------------------------------------
    panel["year"] = panel["qtr"].dt.year
    panel["qtr_num"] = panel["qtr"].dt.quarter
    panel["tq"] = (panel["year"] - 1960) * 4 + (panel["qtr_num"] - 1)

    panel["firm_id"] = panel[firm_col].astype("category").cat.codes + 1
    panel["qtr_str"] = panel["qtr"].astype(str)

    panel = panel.drop(columns="qtr").sort_values(["firm_id", "year", "qtr_num"])

    panel.to_stata(out_path, write_index=False, version=118)

    # --------------------------------------------------
    # 8) Report treatment shares
    # --------------------------------------------------
    n_firms = firm_treat.shape[0]
    n_treated = firm_treat["treated"].sum()
    n_control = n_firms - n_treated

    print(f"Saved: {out_path}")
    print(f"Treated firms (ever hired GenAI): {n_treated} ({n_treated/n_firms:.2%})")
    print(f"Control firms: {n_control} ({n_control/n_firms:.2%})")

    return panel, firm_treat

In [20]:
panel_firm, firm_treat = build_firm_quarter_panel_genai_hiring(
    d_unique=d_unique_filtered,
    recruitment_industries=recruitment_industries,
    date_col="post_date",
    firm_col="company",
    desc_col="description",
    ind_col="rics_k400",
    treat_qtr="2022Q4",
    min_qtr="2022Q1",
    max_qtr="2024Q4",
    out_path="../data/firm_quarter_panel_genai_ever_hired_no_recruit.dta",
)

Excluded recruitment industries: 0 rows
Saved: ../data/firm_quarter_panel_genai_ever_hired_no_recruit.dta
Treated firms (ever hired GenAI): 15568 (41.58%)
Control firms: 21874 (58.42%)


Shifting treatment to 2023Q1

In [15]:
def build_occ_ind_quarter_panel_baseline_no_recruitment(
    d_unique: pd.DataFrame,
    occ_level_path: str,
    recruitment_industries: list,
    exposure_col: str = "human_rating_beta",
    date_col: str = "post_date",
    onet_postings_col: str = "onet_code",
    onet_occ_col: str = "O*NET-SOC Code",
    ind_col: str = "rics_k50",
    recruit_ind_col: str = "rics_k400",
    firm_col: str = "ultimate_parent_company_name",
    company_col: str = "company",  # kept for signature compatibility (unused here)
    exclude_firms: list[str] | None = None,
    treat_date: str = "2022-11-30",                 # for defining pre-period median split
    treat_qtr: str = "2023Q1",                      # NEW: for post indicator timing
    out_path: str = "occ_ind_quarter_panel_highlow_human_beta_no_recruit_baseline.dta",
    min_qtr: str = "2022Q1",
    max_qtr: str | None = None,
):
    d = d_unique.copy()

    # --------------------------------------------------
    # 0) Exclude recruitment-related industry postings
    # --------------------------------------------------
    if recruit_ind_col in d.columns:
        before = len(d)
        d_only_recruit = d[d[recruit_ind_col].isin(recruitment_industries)].copy()
        d = d[~d[recruit_ind_col].isin(recruitment_industries)].copy()
        removed_share = len(d_only_recruit) / before if before else 0
        print(f"Excluded recruitment industries: {len(d_only_recruit)} rows ({removed_share:.2%})")
    else:
        print(f"Warning: '{recruit_ind_col}' not found; no recruitment-industry filtering applied.")

    # --------------------------------------------------
    # 0b) Optional firm exclusion
    # --------------------------------------------------
    if exclude_firms:
        if firm_col not in d.columns:
            print(f"Warning: firm_col '{firm_col}' not found; firm exclusion skipped.")
        else:
            before = len(d)
            d_firm = d[firm_col].astype(str).str.strip().str.casefold()
            exclude_set = {str(x).strip().casefold() for x in exclude_firms}
            mask_excl = d_firm.isin(exclude_set)
            d = d.loc[~mask_excl].copy()
            removed = int(mask_excl.sum())
            print(f"Excluded firms: {removed} rows ({removed / before:.2%})")

    # --------------------------------------------------
    # 1) Dates
    # --------------------------------------------------
    d[date_col] = pd.to_datetime(d[date_col], errors="coerce")
    d = d.dropna(subset=[date_col]).copy()
    treat_dt = pd.Timestamp(treat_date)

    # --------------------------------------------------
    # 2) Load exposure table
    # --------------------------------------------------
    occ = pd.read_csv(occ_level_path)
    if exposure_col not in occ.columns:
        raise ValueError(f"{exposure_col} not found in occ_level.csv")

    d[onet_postings_col] = d[onet_postings_col].astype(str).str.strip()
    occ[onet_occ_col] = occ[onet_occ_col].astype(str).str.strip()
    occ_sub = occ[[onet_occ_col, exposure_col]].rename(columns={onet_occ_col: onet_postings_col})

    d = d.merge(occ_sub, on=onet_postings_col, how="left", indicator=True)
    d = d[d["_merge"] == "both"].drop(columns=["_merge"]).copy()

    # --------------------------------------------------
    # 3) Pre-treatment median (occupation-level, pre-treat_date)
    # --------------------------------------------------
    d_pre = d[d[date_col] < treat_dt]
    occ_pre = d_pre[[onet_postings_col, exposure_col]].drop_duplicates()
    median_pre = float(occ_pre[exposure_col].median())

    occ_treat = (
        d[[onet_postings_col, exposure_col]]
        .drop_duplicates()
        .assign(treated=lambda x: (x[exposure_col] > median_pre).astype(int))
        [[onet_postings_col, "treated"]]
    )

    # --------------------------------------------------
    # 4) Quarter variables
    # --------------------------------------------------
    d["qtr"] = d[date_col].dt.to_period("Q")
    d = d[d["qtr"] >= pd.Period(min_qtr, freq="Q")]
    if max_qtr:
        d = d[d["qtr"] <= pd.Period(max_qtr, freq="Q")]

    # --------------------------------------------------
    # 5) Aggregate to occ–ind–quarter
    # --------------------------------------------------
    panel = (
        d.groupby([onet_postings_col, ind_col, "qtr"])
        .agg(num_postings=("job_id", "count"))
        .reset_index()
    )

    # --------------------------------------------------
    # 6) Balance panel
    # --------------------------------------------------
    occ_ind_pairs = panel[[onet_postings_col, ind_col]].drop_duplicates()
    all_qtrs = pd.period_range(panel["qtr"].min(), panel["qtr"].max(), freq="Q")
    full = (occ_ind_pairs.assign(_tmp=1)
            .merge(pd.DataFrame({"qtr": all_qtrs, "_tmp": 1}), on="_tmp")
            .drop(columns="_tmp"))
    panel = full.merge(panel, on=[onet_postings_col, ind_col, "qtr"], how="left")
    panel["num_postings"] = panel["num_postings"].fillna(0).astype(int)

    # --------------------------------------------------
    # 7) Treatment timing (NOW: treat_qtr inclusive)
    # --------------------------------------------------
    panel = panel.merge(occ_treat, on=onet_postings_col, how="left")
    panel["treated"] = panel["treated"].fillna(0).astype(int)

    TREAT_QTR = pd.Period(treat_qtr, freq="Q")
    panel["post"] = (panel["qtr"] >= TREAT_QTR).astype(int)
    panel["did"] = panel["treated"] * panel["post"]
    panel["log_postings"] = np.log(panel["num_postings"] + 1)

    # --------------------------------------------------
    # 8) Stata-aligned time vars
    # --------------------------------------------------
    panel["year"] = panel["qtr"].dt.year
    panel["qtr_num"] = panel["qtr"].dt.quarter
    panel["tq"] = (panel["year"] - 1960) * 4 + (panel["qtr_num"] - 1)

    base_period = pd.Period(min_qtr, freq="Q")
    base_tq = (base_period.year - 1960) * 4 + (base_period.quarter - 1)

    # relq = 0 at the treatment quarter (so relq=-1 is the quarter right before treatment)
    treat_tq = (TREAT_QTR.year - 1960) * 4 + (TREAT_QTR.quarter - 1)

    panel["t_index"] = panel["tq"] - base_tq
    panel["relq"] = panel["tq"] - treat_tq

    # --------------------------------------------------
    # 9) IDs for FE
    # --------------------------------------------------
    panel["occ_id"] = panel[onet_postings_col].astype("category").cat.codes + 1
    panel["ind_id"] = panel[ind_col].astype("category").cat.codes + 1
    panel["occ_ind_id"] = (panel[onet_postings_col].astype(str) + "|" + panel[ind_col].astype(str)).astype("category").cat.codes + 1

    panel["qtr_str"] = panel["qtr"].astype(str)
    panel = panel.drop(columns="qtr").sort_values(["occ_ind_id", "year", "qtr_num"])

    panel.to_stata(out_path, write_index=False, version=118)

    print(f"Saved: {out_path}")
    print(f"Pre-treatment median ({exposure_col}) using treat_date={treat_date}: {median_pre:.4f}")
    print(f"Post period starts at treat_qtr={treat_qtr} (inclusive)")

    return panel, median_pre


In [17]:
panel, median_pre = build_occ_ind_quarter_panel_baseline_no_recruitment(
    d_unique=d_unique_filtered,
    occ_level_path=r"C:\LocalOneDrive\Documents\Desktop\NUS\Y4S1\FYP\causal-impact-GenAI\data\AI Exposure Scores\occ_level.csv",
    recruitment_industries=recruitment_industries,
    exposure_col="human_rating_beta",
    date_col="post_date",
    onet_postings_col="onet_code",
    onet_occ_col="O*NET-SOC Code",
    ind_col="rics_k50",
    recruit_ind_col="rics_k400",
    firm_col="ultimate_parent_company_name",
    company_col="company",
    exclude_firms=None,
    treat_date="2022-11-30",
    treat_qtr="2022Q3",
    out_path="../data/occ_ind_quarter_panel_highlow_human_beta_no_recruit_baseline_placebo2022Q3.dta",
    min_qtr="2022Q1",
    max_qtr="2024Q4",
)

Excluded recruitment industries: 0 rows (0.00%)
Saved: ../data/occ_ind_quarter_panel_highlow_human_beta_no_recruit_baseline_placebo2022Q3.dta
Pre-treatment median (human_rating_beta) using treat_date=2022-11-30: 0.4062
Post period starts at treat_qtr=2022Q3 (inclusive)
